In [ ]:
# Basic Py6S Workflow

# This script demonstrates how to set up and run a basic 6S simulation using the Py6S library.
# It configures atmospheric parameters, sensor geometry, and wavelength, and then runs the simulation.
# Import necessary modules
import py6s
# Initialize a 6S object
s = py6s.SixS()
# Set the atmospheric model to rural
s.atmos_profile = py6s.AtmosProfile.PredefinedType(py6s.AtmosProfile.Rural)  # FromLatitudeAndDate(latitude, date)  # UserWaterAndOzone(water, ozone)
# Set the visibility to 20 km
s.visibility = 20
# Set the aerosol model to maritime
s.aerosols = py6s.Aerosols.Maritime
# Set the ground reflectance to 0.2
s.ground_reflectance = 0.2
# Set the sensor geometry
s.geometry = py6s.Geometry.User()
# Set the solar zenith angle to 30 degrees
s.geometry.solar_z = 30
# Set the sensor zenith angle to 20 degrees
s.geometry.solar_a = 20
# Set the relative azimuth angle to 10 degrees
s.geometry.relative_a = 10
# Set the wavelength to 0.55 micrometers (550 nm)
s.wavelength = py6s.Wavelength(0.55)


from Py6S import *
import rasterio

# Initialize 6S model
s = SixS()

# Configure atmospheric parameters
s.atmos_profile = AtmosProfile.PredefinedType(AtmosProfile.Tropical)
s.aero_profile = AeroProfile.PredefinedType(AeroProfile.Continental)
s.aot550 = 0.15

# Set sensor geometry (example: Landsat 8)
s.geometry = Geometry.Landsat_TM()
s.geometry.month = 6
s.geometry.day = 15
s.geometry.gmt_decimal_hour = 10.5

# Set wavelength for specific band
s.wavelength = Wavelength(PredefinedWavelengths.LANDSAT_OLI_B2)


In [ ]:
# Atmospheric Correction for Raster Data
def atmospheric_correction(input_raster, output_raster):
    with rasterio.open(input_raster) as src:
        profile = src.profile
        data = src.read(1)

    # Apply 6S model to correct the data
    s.run()
    corrected_data = data * s.outputs.transmittance
    corrected_data = corrected_data.astype(profile['dtype'])

    with rasterio.open(output_raster, 'w', **profile) as dst:
        dst.write(corrected_data, 1)


def correct_pixel(radiance):
    s.atmos_corr = AtmosCorr.AtmosCorrLambertianFromRadiance(radiance)
    s.run()
    return s.outputs.atmos_corrected_reflectance_lambertian

with rasterio.open('input.tif') as src:
    profile = src.profile
    arr = src.read(1)
    
    # Apply correction to each pixel
    corrected = np.vectorize(correct_pixel)(arr)
    
    # Save output
    with rasterio.open('output.tif', 'w', **profile) as dst:
        dst.write(corrected, 1)


In [ ]:
import numpy as np
from py6s import *

# Load metadata
sun_zenith = 30.5  # From metadata XML
earth_sun_dist = 1.015  # Astronomical units
calibration_constant = 0.055  # AWiFS-specific

with rasterio.open('input.tif') as src:
    dn = src.read(1)
    toa_reflectance = (dn * calibration_constant * np.cos(np.radians(sun_zenith))) / (earth_sun_dist ** 2)


In [ ]:
s = SixS()
s.geometry = Geometry.User()
s.geometry.solar_z = sun_zenith
s.geometry.solar_a = metadata['sun_azimuth']
s.geometry.view_z = metadata['view_zenith']
s.geometry.view_a = metadata['view_azimuth']

s.aero_profile = AeroProfile.Continental
s.aot550 = modis_aot  # From Step 2
s.atmos_profile = AtmosProfile.UserWaterAndOzone(modis_h2o, modis_o3)
s.wavelength = Wavelength(PredefinedWavelengths.RESOURCESAT2_AWIFS_B2)


In [ ]:
# Using precomputed 6S coefficients (xa, xb, xc)[9][15]
def atmospheric_correction(toa_ref):
    return (toa_ref - xb) / (xa * (1 + xc * toa_ref))

# Apply to entire array
with rasterio.open('toa_ref.tif') as src:
    profile = src.profile
    arr = src.read(1)
    corrected = atmospheric_correction(arr)
    
# Save output
with rasterio.open('surface_reflectance.tif', 'w', **profile) as dst:
    dst.write(corrected, 1)
